## 6. Apprentissage d'un RNN sur le corpus de texte Victor HUGO

```
Objectif: Prédire le mot suivant à partir des 3 mots précédents, puis un texte mots après mot. Les exemples d'entraînement et la génération utilisent le même format: une fenétre de 3 mots, complétée par des zéros au début si besoin
```

In [1]:
#importing libraries from libs.py

from libs import *        # np, pd, plt, tf, keras, layers, train_test_split, ...

In [2]:
with open('victor-hugo-texts.txt', 'r', encoding='utf-8') as f:
    texts = [line.strip() for line in f if line.strip() != '']
    
print(f"Exemple de texte: {texts[0]}")

Exemple de texte: Mes vers fuiraient


In [3]:
vocab = 1000
window_size = 3

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

In [ ]:
from sklearn.model_selection import train_test_split

# On sépare par LIGNES (et non par fenêtres) : les fenêtres d'une même ligne
# se chevauchent, donc les mélanger entre train et test fausserait l'évaluation.
# Répartition : 80 % train / 10 % validation / 10 % test
seq_train, seq_test = train_test_split(sequences, test_size=0.10, random_state=42)
seq_train, seq_val = train_test_split(seq_train, test_size=1/9, random_state=42)

def make_windows(seqs):
    X, y = [], []
    for seq in seqs:
        for i in range(1, len(seq)):
            X.append(seq[max(0, i - window_size):i])
            y.append(seq[i])
    X = tf.keras.preprocessing.sequence.pad_sequences(
        X, maxlen=window_size, padding="pre"
    )
    return X, np.array(y)

X_train, y_train = make_windows(seq_train)
X_val, y_val = make_windows(seq_val)
X_test, y_test = make_windows(seq_test)

print(f"Train : {X_train.shape}, Validation : {X_val.shape}, Test : {X_test.shape}")


In [5]:
embedding_dim = 16

model = keras.Sequential([
    keras.Input(shape=(window_size,)),
    keras.layers.Embedding(input_dim=vocab, output_dim=embedding_dim),
    keras.layers.SimpleRNN(32),
    keras.layers.Dense(vocab, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)


In [ ]:
# Analyse de performance du model en graph
import matplotlib.pyplot as plt

plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy pendant l'entraînement")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss pendant l'entraînement")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


##### Évaluation sur le test set (Top-1 / Top-5)

In [ ]:
def top_k_scores(X_, y_, k=5):
    probs = model.predict(X_, batch_size=256, verbose=0)
    top1 = np.mean(np.argmax(probs, axis=1) == y_)
    topk_ids = np.argsort(probs, axis=1)[:, -k:]
    topk = np.mean(np.any(topk_ids == y_[:, None], axis=1))
    return top1, topk

for name, (X_, y_) in {"Train": (X_train, y_train),
                       "Validation": (X_val, y_val),
                       "Test": (X_test, y_test)}.items():
    t1, t5 = top_k_scores(X_, y_)
    print(f"{name:<11} Top-1 : {t1:.3f}   Top-5 : {t5:.3f}")

# Référence : toujours prédire le mot le plus fréquent du train
most_common = np.bincount(y_train).argmax()
print(f"\nBaseline (mot le plus fréquent = '{tokenizer.index_word[most_common]}') "
      f"Top-1 sur le test : {np.mean(y_test == most_common):.3f}")


#### Matrix confusion

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Prédictions sur le TEST SET
y_pred = np.argmax(model.predict(X_test, batch_size=256, verbose=0), axis=1)

top_n = 20
freq = np.bincount(y_train, minlength=vocab)          # mots les plus fréquents du train
top_ids = np.argsort(freq)[::-1][:top_n]
labels = [tokenizer.index_word.get(i, str(i)) for i in top_ids]

# on garde les exemples dont le vrai mot est dans le top N ; les autres prédictions deviennent "autres" (-1)
mask = np.isin(y_test, top_ids)
y_true = y_test[mask]
y_hat = np.where(np.isin(y_pred[mask], top_ids), y_pred[mask], -1)

cm = confusion_matrix(y_true, y_hat, labels=list(top_ids) + [-1])[:-1]  # on enlève la ligne vide "autres"
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(12, 8))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels + ["autres"], yticklabels=labels)
plt.xlabel("Mot prédit")
plt.ylabel("Mot réel")
plt.title(f"Matrice de confusion - test set ({top_n} mots les plus fréquents)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
errors = pd.Series(list(zip(
    [tokenizer.index_word.get(i) for i in y_test[y_test != y_pred]],
    [tokenizer.index_word.get(i) for i in y_pred[y_test != y_pred]]
))).value_counts().head(10)
print(errors)


In [ ]:
from sklearn.metrics import classification_report

top_n = 100
freq = np.bincount(y_train, minlength=vocab)
top_ids = np.argsort(freq)[::-1][:top_n]          # 100 mots les plus fréquents du train
names = [tokenizer.index_word.get(i, str(i)) for i in top_ids]

print(classification_report(
    y_test, y_pred,
    labels=top_ids,
    target_names=names,
    zero_division=0
))


In [31]:
def generate_text(start_text, num_words=50, temperature=0.80):
    text = start_text
    for _ in range(num_words):
        #même format qu'à l'entrainement: les 3 derniers mots seulement
        tokens = tokenizer.texts_to_sequences([text])[0][-window_size:]
        tokens = pad_sequences([tokens], maxlen=window_size, padding='pre')
        #probablities du mot suivant
        probs = model.predict(tokens, verbose=0)[0]
        #Tirage pondéré(temperature <1 : plus sûr, >1 : plus varié)
        probs = np.exp(np.log(probs + 1e-9) /temperature)
        probs /= probs.sum()
        index = np.random.choice(len(probs), p=probs)
        
        word = tokenizer.index_word.get(index)
        
        if word is not None:
            text += ' ' +word
            
    return text 

In [ ]:
pad_sequences = tf.keras.preprocessing.sequence.pad_sequences
random_text = generate_text("Ce siècle avait deux ans", num_words=50)
print(random_text)